In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import gensim.downloader as api
from datasets import load_dataset
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.utils import compute_class_weight
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from task6.torch_utils.model_callbacks import EarlyStopping, ModelCheckpoint
from task6.utils.balance_dataset import augment_minority_classes, undersample_majority_classes
from task6.utils.prepare_data import prepare_data
from task6.utils.prepare_rnn_data import prepare_rnn_data

In [13]:
import torch
import torchvision

print("Torch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


Torch version: 2.8.0+cu129
Torchvision version: 0.23.0+cu129
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5080


In [14]:
df = pd.read_csv("../../data/merged_transcripts.csv")
df.head()

,Start Time,End Time,Sentence,Translation,Emotion_fine,Emotion_core,Intensity
0,1900-01-01 00:00:00,1900-01-01 00:00:04,"Ако приемаш моите извинения, просто ме пригърни.","If you accept my apologies, just hug me.",remorse,sadness,moderate
1,1900-01-01 00:00:06,1900-01-01 00:00:08,"Трябвам това, че си ме пригърни.",I need you to hug me.,longing,sadness,moderate
2,1900-01-01 00:00:08,1900-01-01 00:00:11,С теб всеки миски е прекрасен. Обичам.,"With you, every moment is wonderful. I love.",affection,happiness,moderate
3,1900-01-01 00:00:11,1900-01-01 00:00:16,Аз наистина ги гледам по страни и се вълнувам ...,I really look at them from the side and get ex...,excitement,happiness,moderate
4,1900-01-01 00:00:16,1900-01-01 00:00:19,"Ти избра играта, а аз избрах любовта.","You chose the game, and I chose love.",disappointment,sadness,moderate


In [15]:
df, emotions = prepare_data(df, "Translation", "Emotion_core")
print(emotions)
df.head()

Detected dataset type: transcript
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed
['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']


,Start Time,End Time,Sentence,Translation,Emotion_fine,Intensity,ekman_emotion,tokenized_text,lemmatized_text
0,1900-01-01 00:00:00,1900-01-01 00:00:04,"Ако приемаш моите извинения, просто ме пригърни.","If you accept my apologies, just hug me.",remorse,moderate,4,"[if, you, accept, my, apologies, ,, just, hug,...","[if, you, accept, my, apology, ,, just, hug, I..."
1,1900-01-01 00:00:06,1900-01-01 00:00:08,"Трябвам това, че си ме пригърни.",I need you to hug me.,longing,moderate,4,"[i, need, you, to, hug, me, .]","[I, need, you, to, hug, I, .]"
2,1900-01-01 00:00:08,1900-01-01 00:00:11,С теб всеки миски е прекрасен. Обичам.,"With you, every moment is wonderful. I love.",affection,moderate,3,"[with, you, ,, every, moment, is, wonderful, ....","[with, you, ,, every, moment, be, wonderful, ...."
3,1900-01-01 00:00:11,1900-01-01 00:00:16,Аз наистина ги гледам по страни и се вълнувам ...,I really look at them from the side and get ex...,excitement,moderate,3,"[i, really, look, at, them, from, the, side, a...","[I, really, look, at, they, from, the, side, a..."
4,1900-01-01 00:00:16,1900-01-01 00:00:19,"Ти избра играта, а аз избрах любовта.","You chose the game, and I chose love.",disappointment,moderate,4,"[you, chose, the, game, ,, and, i, chose, love...","[you, choose, the, game, ,, and, I, choose, lo..."


In [16]:
df.ekman_emotion.value_counts()

ekman_emotion
6    13360
3     6250
4     3745
0     2205
5     2093
2     1932
1      818
Name: count, dtype: int64

In [17]:
df_test = pd.read_csv("../../data/kinga.csv")
df_test, emotions = prepare_data(df_test, "Translation", "Emotion_core")

Detected dataset type: transcript
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed


In [18]:
target_samples = 1000
upper_limit = 5000

classes_to_augment = df["ekman_emotion"].value_counts()[df["ekman_emotion"].value_counts() < target_samples].index.tolist()
print(f"Classes to augment: {classes_to_augment}")
classes_to_undersample = df["ekman_emotion"].value_counts()[df["ekman_emotion"].value_counts() > upper_limit].index.tolist()
print(f"Classes to undersample: {classes_to_undersample}")

Classes to augment: [1]
Classes to undersample: [6, 3]


In [19]:
# check duplicates in lemmatized_text
print(f"Train duplicates: {df.duplicated(subset=['lemmatized_text']).sum()}")
print(f"Test duplicates: {df_test.duplicated(subset=['lemmatized_text']).sum()}")

Train duplicates: 4032
Test duplicates: 129


In [20]:
df_augmented = augment_minority_classes(df, target_samples=target_samples, classes_to_augment=classes_to_augment)
df_augmented = pd.concat([df, df_augmented]).reset_index(drop=True)
df_augmented = undersample_majority_classes(df_augmented, target_samples=upper_limit, classes_to_reduce=classes_to_undersample)
print(df_augmented["ekman_emotion"].value_counts())
print(f"Original size: {len(df)}, Augmented size: {len(df_augmented)}")
df = df_augmented

Augmenting emotion 1: 818 → 1000
Augmented Sample 1: we do not gather here for you to make relationship
Augmented Sample 2: he say the pizza be doughy on the inside along and greasy .
Augmented Sample 3: this be their love , it be artificial
Reducing emotion 6: 13360 → 5000
Reducing emotion 3: 6250 → 5000
ekman_emotion
6    5000
3    5000
4    3745
0    2205
5    2093
2    1932
1     999
Name: count, dtype: int64
Original size: 30403, Augmented size: 20974


In [21]:
df["lemmatized_text"] = df["lemmatized_text"].apply(lambda tokens: " ".join(tokens))
df_test["lemmatized_text"] = df_test["lemmatized_text"].apply(lambda tokens: " ".join(tokens))

In [22]:
# check duplicates in lemmatized_text
print(f"Train duplicates: {df.duplicated(subset=['lemmatized_text']).sum()}")
print(f"Test duplicates: {df_test.duplicated(subset=['lemmatized_text']).sum()}")

Train duplicates: 2277
Test duplicates: 129


In [23]:
word2vec_model = api.load("word2vec-google-news-300")

In [24]:
X_train, y_train = prepare_rnn_data(df, "lemmatized_text", word2vec_model, "ekman_emotion")
X_test, y_test = prepare_rnn_data(df_test, "lemmatized_text", word2vec_model, "ekman_emotion")
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Preparing RNN data from 20974 samples...
Embedding dimension: 300
Max sequence length: 50


Converting to embeddings: 100%|██████████| 20974/20974 [00:00<00:00, 21216.87it/s]


✓ Final shape: (20974, 50, 300)
✓ OOV rate: 23.09% (40425/175058)
✓ Data type: float64
Preparing RNN data from 791 samples...
Embedding dimension: 300
Max sequence length: 50


Converting to embeddings: 100%|██████████| 791/791 [00:00<00:00, 23970.57it/s]

✓ Final shape: (791, 50, 300)
✓ OOV rate: 27.23% (1711/6283)
✓ Data type: float64
Train shape: (20974, 50, 300), Test shape: (791, 50, 300)


In [25]:
class EmotionDataset(Dataset):
    """PyTorch Dataset for emotion classification"""

    def __init__(self, sequences, labels):
        self.sequences = torch.FloatTensor(sequences)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [26]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
            super().__init__()

            # Input normalization
            self.input_norm = nn.LayerNorm(input_size)

            # BiLSTM with proper initialization
            self.rnn = nn.RNN(
                input_size,
                hidden_size,
                num_layers,
                batch_first=True,
                bidirectional=True,
                dropout=dropout if num_layers > 1 else 0,
            )

            # Attention mechanism (helps with long sequences)
            self.attention = nn.MultiheadAttention(
                embed_dim=hidden_size * 2,
                num_heads=8,
                dropout=dropout,
                batch_first=True
            )

            # Classification layers
            self.dropout = nn.Dropout(dropout)
            self.batch_norm = nn.BatchNorm1d(hidden_size * 2)

            # Simpler classifier
            self.classifier = nn.Sequential(
                nn.Linear(hidden_size * 2, hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_size, num_classes)
            )

            # Initialize weights properly
            self._init_weights()

    def _init_weights(self):
            """Proper weight initialization"""
            for name, param in self.named_parameters():
                if 'weight' in name:
                    if 'rnn' in name:
                        nn.init.xavier_uniform_(param)
                    elif 'linear' in name or 'fc' in name:
                        nn.init.xavier_uniform_(param)
                elif 'bias' in name:
                    nn.init.constant_(param, 0)

    def forward(self, x):
            # Input normalization
            x = self.input_norm(x)

            # RNN
            rnn_out, hidden = self.rnn(x)
            rnn_out = self.dropout(rnn_out)

            # Attention mechanism (instead of just taking last output)
            attn_out, _ = self.attention(rnn_out, rnn_out, rnn_out)

            # Global max pooling across sequence dimension
            pooled = torch.max(attn_out, dim=1)[0]  # Take max across sequence

            # Batch normalization
            pooled = self.batch_norm(pooled)

            # Classification
            output = self.classifier(pooled)

            return output

In [27]:
# train / validation split
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train, shuffle=True)

X_train = np.array(X_train)
y_train = np.array(y_train)
X_val = np.array(X_val)
y_val = np.array(y_val)

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [29]:
num_classes = len(emotions)
epochs = 250
patience = 10
lr_patience = 10
model_save_path = "../../models/RNN/best_model.pth"
num_classes

7

In [30]:
def create_better_training_setup():
    """
    Create better training configuration
    """

    training_config = {
        # Model architecture
        'input_size': 300,
        'hidden_size': 128,  # Smaller hidden size
        'num_layers': 1,     # Fewer layers
        'dropout': 0.2,      # Lower dropout

        # Training parameters
        'learning_rate': 0.001,   # Higher learning rate
        'weight_decay': 1e-5,     # Lower weight decay
        'batch_size': 64,         # Smaller batch size
        'patience': 15,           # More patience
        'lr_patience': 5,         # Reduce LR sooner

        # Class weights (more aggressive for minorities)
        'custom_class_weights': compute_class_weight(
            class_weight='balanced',
            classes=np.arange(num_classes),
            y=y_train
        ).tolist()
    }

    return training_config

In [31]:
def train_rnn(X_train, X_val, X_test, y_train, y_val, y_test, emotions, device, save_path=model_save_path):
    """
    Train BiLSTM with all fixes applied
    """

    print("TRAINING FIXED RNN MODEL")
    print("="*35)

    # Get fixed training configuration
    config = create_better_training_setup()

    # Create datasets with better batch size
    train_dataset = EmotionDataset(X_train, y_train)
    val_dataset = EmotionDataset(X_val, y_val)
    test_dataset = EmotionDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

    # Create fixed model
    model = RNN(
        input_size=config['input_size'],
        hidden_size=config['hidden_size'],
        num_layers=config['num_layers'],
        num_classes=len(emotions),
        dropout=config['dropout']
    ).to(device)

    print(f"Model architecture:")
    print(f"  Hidden size: {config['hidden_size']}")
    print(f"  Num layers: {config['num_layers']}")
    print(f"  Dropout: {config['dropout']}")
    print(f"  Batch size: {config['batch_size']}")

    # Better class weights
    class_weights = torch.FloatTensor([
        config['custom_class_weights'][i] for i in range(len(emotions))
    ]).to(device)

    print(f"Class weights: {class_weights.cpu().numpy()}")

    # Better optimizer and loss
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)  # Add label smoothing
    optimizer = optim.AdamW(  # Use AdamW instead of Adam
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )

    # Better scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )

    print(f"Learning rate: {config['learning_rate']}")
    print(f"Weight decay: {config['weight_decay']}")
    print(f"Using label smoothing: 0.1")

    # Training loop with better monitoring
    best_val_acc = 0
    patience_counter = 0
    overfitting_threshold = 15  # %
    overfitting_patience = 5
    overfitting_counter = 0

    for epoch in range(500):  # Fewer max epochs, focus on quality
        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        if overfitting_counter > overfitting_patience:
            print("⚠️  Stopping training due to overfitting.")
            break

        for sequences, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
            sequences, labels = sequences.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)

            # Check for NaN
            if torch.isnan(loss):
                print("⚠️  NaN loss detected! Stopping training.")
                break

            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        all_val_preds = []
        all_val_labels = []

        with torch.no_grad():
            for sequences, labels in val_loader:
                sequences, labels = sequences.to(device), labels.to(device)
                outputs = model(sequences)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        # Calculate metrics
        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_acc = 100 * train_correct / train_total
        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc = 100 * val_correct / val_total
        current_lr = optimizer.param_groups[0]['lr']

        is_overfitting = epoch_train_acc - epoch_val_acc > overfitting_threshold
        if is_overfitting:
            overfitting_counter += 1
            print(f"⚠️  Overfitting detected! ({overfitting_counter}/{overfitting_patience})")
        else:
            overfitting_counter = 0

        print(f"Epoch {epoch+1:2d}: Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:5.2f}%, "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:5.2f}%, LR: {current_lr:.2e}")

        # Check prediction diversity
        unique_preds = len(set(all_val_preds))
        print(f"           Predicting {unique_preds}/7 different emotions")

        # Early stopping with improvement
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(), save_path)
            print(f"Saved best model with Val Acc: {best_val_acc:.2f}% on epoch {epoch+1}")
        else:
            patience_counter += 1

        if patience_counter >= config['patience']:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Load best model and evaluate
    model.load_state_dict(torch.load(save_path))

    # Final test evaluation
    test_dataset = EmotionDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

    model.eval()
    all_test_preds = []
    all_test_labels = []

    with torch.no_grad():
        for sequences, labels in test_loader:
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences)
            _, predicted = torch.max(outputs, 1)

            all_test_preds.extend(predicted.cpu().numpy())
            all_test_labels.extend(labels.cpu().numpy())

    test_accuracy = sum(p == l for p, l in zip(all_test_preds, all_test_labels)) / len(all_test_preds)

    print(f"\nFINAL FIXED MODEL RESULTS:")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(all_test_labels, all_test_preds, target_names=emotions, zero_division=0, digits=3))

    return model, all_test_preds

In [32]:
 model, preds = train_rnn(X_train, X_val, X_test, y_train, y_val, y_test, emotions, device)

TRAINING FIXED RNN MODEL
Model architecture:
  Hidden size: 128
  Num layers: 1
  Dropout: 0.2
  Batch size: 64
Class weights: [1.359159  2.9995232 1.5506449 0.5992381 0.8001696 1.4313012 0.5992381]
Learning rate: 0.001
Weight decay: 1e-05
Using label smoothing: 0.1


Epoch 1: 100%|██████████| 295/295 [00:01<00:00, 199.05it/s]


Epoch  1: Train Loss: 1.7920, Train Acc: 35.07%, Val Loss: 1.6400, Val Acc: 46.71%, LR: 2.25e-05
           Predicting 7/7 different emotions
Saved best model with Val Acc: 46.71% on epoch 1


Epoch 2: 100%|██████████| 295/295 [00:01<00:00, 288.22it/s]


Epoch  2: Train Loss: 1.6280, Train Acc: 45.82%, Val Loss: 1.5639, Val Acc: 51.43%, LR: 3.90e-05
           Predicting 7/7 different emotions
Saved best model with Val Acc: 51.43% on epoch 2


...

⚠️  Overfitting detected! (4/5)
Epoch 23: Train Loss: 1.0072, Train Acc: 76.43%, Val Loss: 1.7458, Val Acc: 55.00%, LR: 7.59e-04
           Predicting 7/7 different emotions


Epoch 24: 100%|██████████| 295/295 [00:01<00:00, 281.22it/s]


⚠️  Overfitting detected! (5/5)
Epoch 24: Train Loss: 0.9697, Train Acc: 78.47%, Val Loss: 1.6747, Val Acc: 56.39%, LR: 6.77e-04
           Predicting 7/7 different emotions


Epoch 25: 100%|██████████| 295/295 [00:01<00:00, 289.37it/s]

⚠️  Overfitting detected! (6/5)
Epoch 25: Train Loss: 0.9317, Train Acc: 80.57%, Val Loss: 1.7365, Val Acc: 55.53%, LR: 5.90e-04
           Predicting 7/7 different emotions
⚠️  Stopping training due to overfitting.

FINAL FIXED MODEL RESULTS:
Test Accuracy: 0.6195

Classification Report:
              precision    recall  f1-score   support

       anger      0.341     0.400     0.368        35
     disgust      0.208     0.556     0.303         9
        fear      0.520     0.448     0.481        58
         joy      0.606     0.614     0.610       153
     sadness      0.412     0.500     0.452        56
    surprise      0.362     0.507     0.422        67
     neutral      0.805     0.700     0.749       413

    accuracy                          0.619       791
   macro avg      0.465     0.532     0.484       791
weighted avg      0.653     0.619     0.632       791

